In [126]:
import os, re, json, uuid, asyncio, time, requests, httpx
from datetime import datetime
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from google import genai
from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

load_dotenv()
gen = genai.Client()

class CONFIG:
    MODEL          = "gemini-3.1-flash-lite"
    KAKAO_SIZE     = 15
    MIN_PLACES     = 5
    BLOG_PER_PLACE = 5
    MAX_REVIEW     = 20
    NEWS_DISPLAY   = 10
    JUDGES         = 2
    TOP_N          = 8
    TOOL_DIR       = Path("./tool_file")
    BLOG_EXTRACT      = 10    # 5 → 10 (블로그 더 봐서 이름 많이 확보)
    MAX_EXTRACT_NAMES = 10    # 추출 이름 상한 (새로 추가)

CONFIG.TOOL_DIR.mkdir(exist_ok=True)

def log(who, msg): print(f"  🔧[{who}] {msg}")

def dump(who, tag, data) -> str:
    f = CONFIG.TOOL_DIR / f"{datetime.now():%H%M%S%f}_{who}_{tag}.json"
    f.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return str(f)

def load(path) -> list[dict]:
    """앞 툴이 떨군 file을 매장 리스트로 정규화. 카카오/블로그추출 형태 자동 판별."""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(data, dict) and "places" in data:
        return data["places"]
    if isinstance(data, list) and data and isinstance(data[0], dict):
        return data
    if isinstance(data, list):
        return [{"name": n} for n in data]
    return []

In [117]:
class ReviewSummary(BaseModel):
    query_relevance: str = Field(description="질문과 관련된 내용. 원문에 없으면 '관련 언급 없음'")
    constraints: str     = Field(description="질문의 제약조건(반려동물·바다뷰·주차 등) 충족 여부. 근거 있으면 인용, 없으면 '후기에서 확인 불가'")
    hours: str           = Field(description="영업시간. 원문에 있으면만, 없으면 '정보 없음'")
    strengths: str       = Field(description="장점·특징")
    weaknesses: str      = Field(description="단점. 없으면 '언급 없음'")

class ExtractedPlaces(BaseModel):
    names: list[str] = Field(description="원문에 실제 등장한 장소 상호명만. 지어내거나 일반명사('바다 캠핑장') 금지, 고유 상호만.")

class TopPicks(BaseModel):
    picks: list[str] = Field(description="질문에 가장 맞는 장소명 top-n. 후보에 있는 이름만, 좋은 순서대로.")

class Recommendation(BaseModel):
    name: str
    address: str = Field(description="주소. 없으면 '정보 없음'")
    phone: str   = Field(description="전화번호. 없으면 '정보 없음'")
    hours: str   = Field(description="영업시간. 근거 없으면 '정보 없음'")
    reason: str  = Field(description="추천 이유. 리뷰 요약 근거로")

class Result(BaseModel):
    recommendations: list[Recommendation]

EMPTY_REVIEW = {"query_relevance": "", "constraints": "", "hours": "",
                "strengths": "", "weaknesses": "", "review_count": 0}

In [118]:
def _kakao_raw(query, code=""):
    params = {"query": query, "size": CONFIG.KAKAO_SIZE}
    if code: params["category_group_code"] = code
    r = requests.get("https://dapi.kakao.com/v2/local/search/keyword.json",
        headers={"Authorization": f"KakaoAK {os.environ['KAKAO_REST_API_KEY']}"},
        params=params, timeout=5)
    r.raise_for_status()
    return [{"name": d["place_name"], "category": d["category_name"],
             "address": d["road_address_name"] or d["address_name"],
             "phone": d["phone"], "x": d["x"], "y": d["y"], "url": d["place_url"]}
            for d in r.json()["documents"]]

CATEGORY = {"카페": "CE7", "음식점": "FD6", "관광명소": "AT4", "문화시설": "CT1"}

def search_places(queries: list[str], category: str = "") -> dict:
    """카카오로 고정 장소(카페·음식점·관광지·명소·미술관)의 팩트를 찾아 파일로 저장한다.
    여러 검색어로 검색·중복제거하고, 부족하면 지역 단위로 넓혀 확보한다.
    결과가 부족하면 enough=false로 알린다(→ search_places_from_blog 고려).

    Args:
        queries: 검색어 리스트('지역+짧은 키워드' 3단어 이내, 연도 금지).
        category: 선택. '카페','음식점','관광명소','문화시설'.
    Returns:
        {count, enough, file}. file을 search_blog_reviews에 그대로 넘겨라.
    """
    log("kakao", f"검색 {queries} cat={category}")
    code = CATEGORY.get(category, "")
    seen, places = set(), []
    def add(q):
        for d in _kakao_raw(q, code):
            if d["name"] not in seen:
                seen.add(d["name"]); places.append(d)
    for q in queries: add(q)
    if len(places) < CONFIG.MIN_PLACES:
        for q in queries:
            add(q.split()[0] if q.split() else q)
            if len(places) >= CONFIG.MIN_PLACES: break
    enough = len(places) >= CONFIG.MIN_PLACES
    log("kakao", f"→ {len(places)}건 (enough={enough})")
    return {"count": len(places), "enough": enough,
            "file": dump("kakao", "places", places)}

In [119]:
async def _blog_links(query, http, k):
    r = await http.get("https://openapi.naver.com/v1/search/blog.json",
        headers={"X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
                 "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"]},
        params={"query": query, "display": 15, "sort": "sim"}, timeout=5)
    clean = lambda s: s.replace("<b>", "").replace("</b>", "")
    key = query.split()[0]
    links = []
    for it in r.json()["items"]:
        if "blog.naver.com" in it["link"] and (
            key in clean(it["title"]) or key in clean(it["description"]) or query in clean(it["title"])):
            links.append(it["link"])
        if len(links) >= k: break
    return links

async def _crawl(url, http):
    h = {"User-Agent": "Mozilla/5.0"}
    shell = await http.get(url, headers=h, timeout=5)
    fr = BeautifulSoup(shell.text, "html.parser").select_one("iframe#mainFrame")
    if not fr: return ""
    real = await http.get("https://blog.naver.com" + fr["src"], headers=h, timeout=5)
    soup = BeautifulSoup(real.text, "html.parser")
    c = soup.select_one(".se-main-container") or soup.select_one("#postViewArea")
    return c.get_text("\n", strip=True) if c else ""

async def _summarize(name, merged, query):
    prompt = (f"장소명:'{name}'\n질문:'{query}'\n\n"
              f"질문 관련 정보를 우선 요약하되 원문에 없는 건 지어내지 마라. "
              f"여러 후기 공통 내용을 우선하라.\n\n--- 원문 ---\n{merged}")
    r = await gen.aio.models.generate_content(model=CONFIG.MODEL, contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": ReviewSummary})
    return r.parsed

async def _process(place, query, http):
    try:
        links = await _blog_links(place["name"], http, CONFIG.BLOG_PER_PLACE)
        if not links:
            return {**place, **EMPTY_REVIEW, "review_status": "블로그 없음"}
        raws = await asyncio.gather(*[_crawl(u, http) for u in links])
        valid = [x for x in raws if len(x) >= 100]
        if not valid:
            return {**place, **EMPTY_REVIEW, "review_status": "본문 수집 실패"}
        s = await _summarize(place["name"], "\n\n===\n\n".join(valid), query)
        return {**place, **s.model_dump(), "review_count": len(valid), "review_status": "ok"}
    except Exception as e:
        return {**place, **EMPTY_REVIEW, "review_status": f"에러:{type(e).__name__}"}

In [ ]:
async def search_places_from_blog(query: str) -> dict:
    """카카오가 부족한 장소(캠핑장·펜션·계곡 등)를 네이버 블로그 원문에서 찾아 실제 상호명을 추출·저장한다.
    search_places 결과가 enough=false일 때 쓴다.

    Args:
        query: '지역 + 장소 종류'. 예: '마산 바다 캠핑장'.
    Returns:
        {names, file}. file을 search_blog_reviews에 넘겨라.
    """
    log("blog_extract", f"'{query}' 장소명 추출")
    async with httpx.AsyncClient() as http:
        links = await _blog_links(query, http, CONFIG.BLOG_EXTRACT)
        if not links:
            return {"names": [], "file": dump("blog_extract", "places", [])}
        raws = await asyncio.gather(*[_crawl(u, http) for u in links])
        valid = [x for x in raws if len(x) >= 100]
        if not valid:
            return {"names": [], "file": dump("blog_extract", "places", [])}
        prompt = (f"질문:'{query}'\n아래 블로그 원문에서 실제 언급된 장소 상호명을 최대 10개까지 뽑아라. "
                  f"질문과 무관한 곳(예: 캠핑장 찾는데 카페)은 제외하고, 원문에 없는 건 절대 지어내지 마라.\n\n--- 원문 ---\n" + "\n\n===\n\n".join(valid))
        r = await gen.aio.models.generate_content(model=CONFIG.MODEL, contents=prompt,
            config={"response_mime_type": "application/json", "response_schema": ExtractedPlaces})
    names = list(dict.fromkeys(r.parsed.names))[:CONFIG.MAX_EXTRACT_NAMES]
    log("blog_extract", f"→ {names}")
    return {"names": names,
            "file": dump("blog_extract", "places", [{"name": n} for n in names])}

async def search_blog_reviews(source_file: str, query: str) -> dict:
    """앞 툴(search_places 또는 search_places_from_blog)이 저장한 file을 받아,
    그 안의 모든 장소의 네이버 블로그 후기를 병렬 크롤링·요약한다.
    매장 리스트를 직접 만들지 말고 반드시 앞 툴의 file 경로를 넘겨라.

    Args:
        source_file: search_places / search_places_from_blog가 반환한 file 경로.
        query: 사용자 질문. 요약이 이 관점을 우선하게 한다.
    Returns:
        {count, file}. file을 ensemble_judge에 넘겨라.
    """
    places = load(source_file)[:CONFIG.MAX_REVIEW]
    log("blog_review", f"{len(places)}개 장소 병렬 요약 ({Path(source_file).name})")
    async with httpx.AsyncClient() as http:
        enriched = await asyncio.gather(*[_process(p, query, http) for p in places])
    ok = sum(1 for p in enriched if p.get("review_status") == "ok")
    log("blog_review", f"→ 요약 {ok}/{len(enriched)}건")
    return {"count": len(enriched), "file": dump("blog_review", "reviews", enriched)}

In [121]:
def search_news(query: str) -> dict:
    """네이버 뉴스로 시간 기반 이벤트(축제·팝업·콘서트·전시)를 최신순으로 찾아 저장한다.
    검색어에 연도를 넣지 마라. 고정 장소엔 쓰지 마라.

    Args:
        query: '지역 + 이벤트 종류'. 예: '부산 축제'.
    Returns:
        {today, count, file}
    """
    log("news", f"'{query}'")
    r = requests.get("https://openapi.naver.com/v1/search/news.json",
        headers={"X-Naver-Client-Id": os.environ["NAVER_CLIENT_ID"],
                 "X-Naver-Client-Secret": os.environ["NAVER_CLIENT_SECRET"]},
        params={"query": query, "display": CONFIG.NEWS_DISPLAY, "sort": "date"}, timeout=5)
    clean = lambda s: s.replace("<b>", "").replace("</b>", "").replace("&quot;", '"')
    news = [{"title": clean(it["title"]), "summary": clean(it["description"]),
             "date": it["pubDate"], "link": it["link"]} for it in r.json()["items"]]
    return {"today": datetime.now().strftime("%Y-%m-%d"), "count": len(news),
            "file": dump("news", "news", news)}

In [122]:
async def _judge_once(candidates, query):
    lines = [f"- {c['name']}: {(c.get('query_relevance') or c.get('strengths') or '(리뷰 정보 없음)')[:200]}"
             for c in candidates]
    prompt = (f"질문:'{query}'\n\n후보 중 가장 맞는 {CONFIG.TOP_N}곳을 좋은 순서로 골라라"
              f"(후보 이름만):\n" + "\n".join(lines))
    r = await gen.aio.models.generate_content(model=CONFIG.MODEL, contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": TopPicks})
    return r.parsed.picks

async def ensemble_judge(reviews_file: str, query: str) -> dict:
    """search_blog_reviews가 저장한 file을 받아, 판단 모델 여러 개를 병렬 실행하고
    득표수로 리랭킹해 top-n 후보를 팩트와 함께 저장한다.

    Args:
        reviews_file: search_blog_reviews가 반환한 file 경로.
        query: 사용자 질문.
    Returns:
        {picks(팩트+votes), votes, file}
    """
    candidates = load(reviews_file)
    log("judge", f"후보 {len(candidates)}개 → {CONFIG.JUDGES}개 모델 병렬")
    results = await asyncio.gather(*[_judge_once(candidates, query) for _ in range(CONFIG.JUDGES)])
    votes = Counter(name for picks in results for name in picks)
    by_name = {c["name"]: c for c in candidates}
    picked = [{**by_name[nm], "votes": v} for nm, v in votes.most_common(CONFIG.TOP_N) if nm in by_name]
    log("judge", f"→ {[(p['name'], p['votes']) for p in picked]}")
    return {"picks": picked, "votes": dict(votes),
            "file": dump("judge", "picks", picked)}

In [123]:
def build_instruction(ctx) -> str:
    today = datetime.now().strftime("%Y년 %m월 %d일")
    return (
        f"오늘은 {today}이다. 날짜 질문은 이 기준으로 계산하고, 검색어에 연도를 넣지 마라.\n\n"
        "너는 지역 추천 에이전트다. 각 툴은 결과를 파일로 저장하고 file 경로를 돌려준다. "
        "너는 매장 리스트를 직접 만들지 말고, 앞 툴이 준 file 경로를 다음 툴에 그대로 넘기기만 한다.\n\n"
        "장소 추천 흐름:\n"
        "1) 카페·음식점·관광지 등은 search_places로 검색한다(검색어 여러 각도로 증강).\n"
        "2) 결과가 enough=false면(캠핑장·펜션 등) search_places_from_blog로 장소를 찾는다.\n"
        "3) 위에서 실제로 쓴 툴의 file을 search_blog_reviews에 넘긴다.\n"
        "4) search_blog_reviews의 file을 ensemble_judge에 넘긴다.\n"
        "5) ensemble_judge의 picks를 근거로 추천을 정리한다.\n\n"
        "축제·공연 등 이벤트는 search_news를 쓴다.\n"
        "규칙: 장소명을 지어내지 마라. 조건에 맞는 결과가 없으면 정직히 '찾지 못했다'고 답하라."
    )

agent = LlmAgent(
    name="local_recommender",
    model=CONFIG.MODEL,
    instruction=build_instruction,
    tools=[search_places, search_places_from_blog, search_blog_reviews, search_news, ensemble_judge],
    output_schema=Result,
)

In [125]:
def _short(o, n=200):
    s = json.dumps(o, ensure_ascii=False, default=str)
    return s if len(s) <= n else s[:n] + f"...({len(s)}자)"

async def ask(text: str):
    runner = InMemoryRunner(agent=agent, app_name="app")
    sid = str(uuid.uuid4())
    await runner.session_service.create_session(app_name="app", user_id="u1", session_id=sid)
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    print(f"\n{'='*60}\n질문: {text}\n{'='*60}")
    final, t = "", time.time()
    async for event in runner.run_async(user_id="u1", session_id=sid, new_message=msg):
        for c in event.get_function_calls():
            print(f"\n▶ {c.name}  IN: {_short(dict(c.args))}")
        for r in event.get_function_responses():
            b = r.response
            if isinstance(b, dict):
                brief = {k: v for k, v in b.items() if k != "picks"}
                if "picks" in b:
                    brief["picks"] = [(p["name"], p.get("votes")) for p in b["picks"]]
                print(f"   OUT {r.name}: {_short(brief)}")
        if event.is_final_response() and event.content:
            final = event.content.parts[0].text
    print(f"\n⏱️ {time.time()-t:.1f}초")
    cleaned = re.sub(r"^```(?:json)?|```$", "", final.strip()).strip()
    try:
        data = json.loads(cleaned)
        print(json.dumps(data, ensure_ascii=False, indent=2))
        return data
    except json.JSONDecodeError:
        print("[파싱 실패]\n", final); return None

#await ask("마산 귀산에서 오션뷰 커피맛집을 찾아줘")
await ask("마산 근처 바다 보이고 반려동물 가능한 캠핑장 찾아줘")


질문: 마산 근처 바다 보이고 반려동물 가능한 캠핑장 찾아줘

▶ search_places_from_blog  IN: {"query": "마산 바다 근처 반려동물 동반 캠핑장"}
  🔧[blog_extract] '마산 바다 근처 반려동물 동반 캠핑장' 장소명 추출
  🔧[blog_extract] → ['타이거스피드 안녕오토캠핑장', '카페가포로702']
   OUT search_places_from_blog: {"names": ["타이거스피드 안녕오토캠핑장", "카페가포로702"], "file": "tool_file\\194339757702_blog_extract_places.json"}

▶ search_blog_reviews  IN: {"source_file": "tool_file\\194339757702_blog_extract_places.json", "query": "마산 바다 근처 반려동물 가능한 캠핑장"}
  🔧[blog_review] 2개 장소 병렬 요약 (194339757702_blog_extract_places.json)
  🔧[blog_review] → 요약 1/2건
   OUT search_blog_reviews: {"count": 2, "file": "tool_file\\194342839756_blog_review_reviews.json"}

▶ ensemble_judge  IN: {"query": "마산 바다 근처 반려동물 가능한 캠핑장", "reviews_file": "tool_file\\194342839756_blog_review_reviews.json"}
  🔧[judge] 후보 2개 → 2개 모델 병렬
  🔧[judge] → [('타이거스피드 안녕오토캠핑장', 2)]
   OUT ensemble_judge: {"votes": {"타이거스피드 안녕오토캠핑장": 2}, "file": "tool_file\\194344513512_judge_picks.json", "picks": [["타이거스피드 안녕오토캠핑장", 2]]}

▶ sea

{'recommendations': [{'name': '타이거스피드 안녕오토캠핑장',
   'address': '경상남도 창원시 마산합포구 구산면 유산군령로 258-20',
   'phone': '정보 없음',
   'hours': '입실 13:00 / 퇴실 12:00, 매너타임 23:00~07:00',
   'reason': '마산 구산면에 위치한 바다 바로 앞 오토캠핑장으로 탁 트인 오션뷰가 큰 장점입니다. 반려동물 동반이 가능하며 시설이 깔끔하고 정수기, 전자레인지 등 편의시설이 잘 갖춰져 있어 캠퍼들에게 인기가 많습니다. 사이트 옆 주차도 가능하여 편리합니다.'}]}